# CSV NaN Pinpoint Script
Find line numbers where NaN values occur in a specified column of a CSV file.

# Find NaN Records in CSV Column
This cell contains the script to find line numbers where NaN values occur in a specified column of a CSV file.

In [ ]:
import pandas as pd
import argparse
import sys

def find_nan_lines(csv_file, column_name):
    """
    Find line numbers where NaN values occur in a specified column

    Args:
        csv_file (str): Path to the CSV file
        column_name (str): Name of the column to check for NaN values

    Returns:
        list: Line numbers (1-indexed) containing NaN values
    """
    try:
        # Read the CSV file
        df = pd.read_csv(csv_file)

        # Check if column exists
        if column_name not in df.columns:
            print(f"Error: Column '{column_name}' not found in the CSV file.")
            print(f"Available columns: {', '.join(df.columns)}")
            return []

        # Find rows with NaN values in the specified column
        nan_mask = df[column_name].isna()
        nan_rows = df[nan_mask]

        # Get line numbers (adding 2 because pandas index is 0-based and we need to account for header)
        nan_lines = (nan_rows.index + 2).tolist()

        return nan_lines

    except FileNotFoundError:
        print(f"Error: File '{csv_file}' not found.")
        return []
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return []

# Example usage:
# nan_lines = find_nan_lines('your.csv', 'column_name')
# print(nan_lines)

# CSV Features Validation Script
Check all CSV files in a folder for records (non-empty files).

In [ ]:
import os
import pandas as pd
import glob

def validate_csv_records(folder_path):
    """
    Check all CSV files in folder and subfolders for records (non-empty files).

    Args:
        folder_path (str): Path to the folder to scan

    Returns:
        dict: Dictionary with file paths as keys and record counts as values
    """

    empty_files = {}
    valid_files = {}
    error_files = {}

    # Find all CSV files recursively
    csv_pattern = os.path.join(folder_path, '**', '*.csv')
    csv_files = glob.glob(csv_pattern, recursive=True)

    print(f"Found {len(csv_files)} CSV files to validate...")

    for csv_file in csv_files:
        try:
            # Read the CSV file and check number of records
            df = pd.read_csv(csv_file)
            record_count = len(df)

            if record_count == 0:
                empty_files[csv_file] = record_count
            else:
                valid_files[csv_file] = record_count

        except Exception as e:
            error_files[csv_file] = str(e)

    # Print results
    print(f"\nValidation Results:")
    print(f"- Files with records: {len(valid_files)}")
    print(f"- Empty files: {len(empty_files)}")
    print(f"- Error files: {len(error_files)}")

    if empty_files:
        print(f"\nFiles with no records:")
        for file_path in empty_files.keys():
            print(f"  {file_path}")

    if error_files:
        print(f"\nFiles with read errors:")
        for file_path, error in error_files.items():
            print(f"\n{file_path}")
            print(f"  Error: {error}")

    return {
        'valid_files': valid_files,
        'empty_files': empty_files,
        'error_files': error_files
    }

def main():
    """Main function to run the validation."""
    # Set the folder path to scan
    folder_path = r"1d-2005"

    if not folder_path:
        folder_path = os.getcwd()

    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return

    print(f"Scanning folder: {folder_path}")
    results = validate_csv_records(folder_path)

    # Save results to a text file
    output_file = os.path.join(os.path.dirname(
        __file__), 'record_validation_results.txt')
    with open(output_file, 'w') as f:
        f.write(f"CSV Record Validation Results\n")
        f.write(f"Scanned folder: {folder_path}\n")
        f.write(f"Files with records: {len(results['valid_files'])}\n")
        f.write(f"Empty files: {len(results['empty_files'])}\n")
        f.write(f"Error files: {len(results['error_files'])}\n\n")

        if results['valid_files']:
            f.write("Files with records:\n")
            for file_path, record_count in results['valid_files'].items():
                f.write(f"{file_path} - {record_count} records\n")

        if results['empty_files']:
            f.write("\nFiles with no records:\n")
            for file_path in results['empty_files'].keys():
                f.write(f"{file_path}\n")

        if results['error_files']:
            f.write("\nFiles with read errors:\n")
            for file_path, error in results['error_files'].items():
                f.write(f"\n{file_path}\n")
                f.write(f"Error: {error}\n")

    print(f"\nResults saved to: {output_file}")

if __name__ == "__main__":
    main()

# CSV Records Validation Script
Check all CSV files in a folder for required columns.

In [ ]:
import os
import pandas as pd
import glob

def validate_csv_columns(folder_path):
    """
    Check all CSV files in folder and subfolders for required columns.

    Args:
        folder_path (str): Path to the folder to scan

    Returns:
        dict: Dictionary with file paths as keys and missing columns as values
    """

    # Required columns
    required_columns = [
        'date', 'open', 'high', 'low', 'close', 'volume', 'rate', 'middle', 'tp', 'boll',
        'boll_ub', 'boll_lb', 'macd', 'macds', 'macdh', 'pvo', 'pvos', 'pvoh', 'ppo',
        'ppos', 'ppoh', 'qqe', 'qqel', 'qqes', 'cr', 'cr-ma1', 'cr-ma2', 'cr-ma3',
        'tr', 'dx', 'adx', 'adxr', 'log-ret', 'wt1', 'wt2', 'supertrend_ub',
        'supertrend_lb', 'supertrend', 'bop', 'cti', 'eribull', 'eribear', 'rvgi',
        'rvgis', 'kst', 'num', 'ao', 'aroon', 'atr', 'cci', 'change', 'chop', 'cmo',
        'coppock', 'dma', 'ichimoku', 'inertia', 'ftr', 'kama', 'kdjk', 'kdjd', 'kdjj',
        'ker', 'mfi', 'ndi', 'pdi', 'pgo', 'psl', 'rsi', 'rsv', 'stochrsi', 'tema',
        'trix', 'wr', 'vr', 'vwma', 'close_10_ema', 'close_10_sma'
    ]

    required_columns_set = set(required_columns)
    invalid_files = {}
    valid_files = []
    error_files = {}

    # Find all CSV files recursively
    csv_pattern = os.path.join(folder_path, '**', '*.csv')
    csv_files = glob.glob(csv_pattern, recursive=True)

    print(f"Found {len(csv_files)} CSV files to validate...")

    for csv_file in csv_files:
        try:
            # Read only the header to check columns
            df = pd.read_csv(csv_file, nrows=0)
            file_columns_set = set(df.columns)

            # Check for missing columns
            missing_columns = required_columns_set - file_columns_set

            if missing_columns:
                invalid_files[csv_file] = sorted(list(missing_columns))
            else:
                valid_files.append(csv_file)

        except Exception as e:
            error_files[csv_file] = str(e)

    # Print results
    print(f"\nValidation Results:")
    print(f"- Valid files: {len(valid_files)}")
    print(f"- Invalid files: {len(invalid_files)}")
    print(f"- Error files: {len(error_files)}")

    if invalid_files:
        print(f"\nFiles missing required columns:")
        for file_path, missing_cols in invalid_files.items():
            print(f"\n{file_path}")
            print(
                f"  Missing columns ({len(missing_cols)}): {', '.join(missing_cols)}")

    if error_files:
        print(f"\nFiles with read errors:")
        for file_path, error in error_files.items():
            print(f"\n{file_path}")
            print(f"  Error: {error}")

    return {
        'valid_files': valid_files,
        'invalid_files': invalid_files,
        'error_files': error_files
    }

def main():
    """Main function to run the validation."""
    # Set the folder path to scan
    folder_path = r"1d-2005"

    if not folder_path:
        folder_path = os.getcwd()

    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return

    print(f"Scanning folder: {folder_path}")
    results = validate_csv_columns(folder_path)

    # Save results to a text file
    output_file = os.path.join(os.path.dirname(
        __file__), 'validation_results.txt')
    with open(output_file, 'w') as f:
        f.write(f"CSV Validation Results\n")
        f.write(f"Scanned folder: {folder_path}\n")
        f.write(f"Valid files: {len(results['valid_files'])}\n")
        f.write(f"Invalid files: {len(results['invalid_files'])}\n")
        f.write(f"Error files: {len(results['error_files'])}\n\n")

        if results['invalid_files']:
            f.write("Files missing required columns:\n")
            for file_path, missing_cols in results['invalid_files'].items():
                f.write(f"\n{file_path}\n")
                f.write(
                    f"Missing columns ({len(missing_cols)}): {', '.join(missing_cols)}\n")

        if results['error_files']:
            f.write("\nFiles with read errors:\n")
            for file_path, error in results['error_files'].items():
                f.write(f"\n{file_path}\n")
                f.write(f"Error: {error}\n")

    print(f"\nResults saved to: {output_file}")

if __name__ == "__main__":
    main()

# PKL Records Validation Script
Check all PKL files in a folder for required columns and empty files.

In [ ]:
import os
import pandas as pd
import glob

def validate_pkl_columns(folder_path):
    """
    Check all PKL files in folder and subfolders for required columns.

    Args:
        folder_path (str): Path to the folder to scan

    Returns:
        dict: Dictionary with file paths as keys and missing columns as values
    """

    required_columns = [
        'open', 'high', 'low', 'close', 'volume', 'rate', 'middle', 'tp', 'boll',
        'boll_ub', 'boll_lb', 'macd', 'macds', 'macdh', 'pvo', 'pvos', 'pvoh', 'ppo',
        'ppos', 'ppoh', 'qqe', 'qqel', 'qqes', 'cr', 'cr-ma1', 'cr-ma2', 'cr-ma3',
        'tr', 'dx', 'adx', 'adxr', 'log-ret', 'wt1', 'wt2', 'supertrend_ub',
        'supertrend_lb', 'supertrend', 'bop', 'cti', 'eribull', 'eribear', 'rvgi',
        'rvgis', 'kst', 'num', 'ao', 'aroon', 'atr', 'cci', 'change', 'chop', 'cmo',
        'coppock', 'dma', 'ichimoku', 'inertia', 'ftr', 'kama', 'kdjk', 'kdjd', 'kdjj',
        'ker', 'mfi', 'ndi', 'pdi', 'pgo', 'psl', 'rsi', 'rsv', 'stochrsi', 'tema',
        'trix', 'wr', 'vr', 'vwma', 'close_10_ema', 'close_10_sma'
    ]

    required_columns_set = set(required_columns)
    invalid_files = {}
    valid_files = []
    empty_files = []
    error_files = {}

    pkl_pattern = os.path.join(folder_path, '**', '*.pkl')
    pkl_files = glob.glob(pkl_pattern, recursive=True)

    print(f"Found {len(pkl_files)} PKL files to validate...")

    for pkl_file in pkl_files:
        try:
            df = pd.read_pickle(pkl_file)
            if len(df) == 0:
                empty_files.append(pkl_file)
                continue
            file_columns_set = set(df.columns)
            missing_columns = required_columns_set - file_columns_set
            if missing_columns:
                invalid_files[pkl_file] = sorted(list(missing_columns))
            else:
                valid_files.append(pkl_file)
                print(f"✓ {os.path.basename(pkl_file)}: {len(df)} rows, columns: {list(df.columns)}")
        except Exception as e:
            error_files[pkl_file] = str(e)

    print(f"\nValidation Results:")
    print(f"- Valid files: {len(valid_files)}")
    print(f"- Invalid files: {len(invalid_files)}")
    print(f"- Empty files (no records): {len(empty_files)}")
    print(f"- Error files: {len(error_files)}")

    if empty_files:
        print(f"\nFiles with NO RECORDS:")
        for file_path in empty_files:
            print(f"  {file_path}")

    if invalid_files:
        print(f"\nFiles missing required columns:")
        for file_path, missing_cols in invalid_files.items():
            print(f"\n{file_path}")
            print(f"  Missing columns ({len(missing_cols)}): {', '.join(missing_cols)}")

    if error_files:
        print(f"\nFiles with read errors:")
        for file_path, error in error_files.items():
            print(f"\n{file_path}")
            print(f"  Error: {error}")

    return {
        'valid_files': valid_files,
        'invalid_files': invalid_files,
        'empty_files': empty_files,
        'error_files': error_files
    }

# Example usage:
# results = validate_pkl_columns('your_folder')

# PKL Features Validation Script
Check all PKL files in a folder for required columns.

In [ ]:
import os
import pandas as pd
import glob

def validate_pkl_columns_features(folder_path):
    """
    Check all PKL files in folder and subfolders for required columns.

    Args:
        folder_path (str): Path to the folder to scan

    Returns:
        dict: Dictionary with file paths as keys and missing columns as values
    """

    required_columns = [
        'open', 'high', 'low', 'close', 'volume', 'rate', 'middle', 'tp', 'boll',
        'boll_ub', 'boll_lb', 'macd', 'macds', 'macdh', 'pvo', 'pvos', 'pvoh', 'ppo',
        'ppos', 'ppoh', 'qqe', 'qqel', 'qqes', 'cr', 'cr-ma1', 'cr-ma2', 'cr-ma3',
        'tr', 'dx', 'adx', 'adxr', 'log-ret', 'wt1', 'wt2', 'supertrend_ub',
        'supertrend_lb', 'supertrend', 'bop', 'cti', 'eribull', 'eribear', 'rvgi',
        'rvgis', 'kst', 'num', 'ao', 'aroon', 'atr', 'cci', 'change', 'chop', 'cmo',
        'coppock', 'dma', 'ichimoku', 'inertia', 'ftr', 'kama', 'kdjk', 'kdjd', 'kdjj',
        'ker', 'mfi', 'ndi', 'pdi', 'pgo', 'psl', 'rsi', 'rsv', 'stochrsi', 'tema',
        'trix', 'wr', 'vr', 'vwma', 'close_10_ema', 'close_10_sma'
    ]

    required_columns_set = set(required_columns)
    invalid_files = {}
    valid_files = []
    error_files = {}

    pkl_pattern = os.path.join(folder_path, '**', '*.pkl')
    pkl_files = glob.glob(pkl_pattern, recursive=True)

    print(f"Found {len(pkl_files)} PKL files to validate...")

    for pkl_file in pkl_files:
        try:
            df = pd.read_pickle(pkl_file)
            file_columns_set = set(df.columns)
            missing_columns = required_columns_set - file_columns_set
            if missing_columns:
                invalid_files[pkl_file] = sorted(list(missing_columns))
            else:
                valid_files.append(pkl_file)
                print(f"✓ {os.path.basename(pkl_file)}: {len(df)} rows, columns: {list(df.columns)}")
        except Exception as e:
            error_files[pkl_file] = str(e)

    print(f"\nValidation Results:")
    print(f"- Valid files: {len(valid_files)}")
    print(f"- Invalid files: {len(invalid_files)}")
    print(f"- Error files: {len(error_files)}")

    if invalid_files:
        print(f"\nFiles missing required columns:")
        for file_path, missing_cols in invalid_files.items():
            print(f"\n{file_path}")
            print(f"  Missing columns ({len(missing_cols)}): {', '.join(missing_cols)}")

    if error_files:
        print(f"\nFiles with read errors:")
        for file_path, error in error_files.items():
            print(f"\n{file_path}")
            print(f"  Error: {error}")

    return {
        'valid_files': valid_files,
        'invalid_files': invalid_files,
        'error_files': error_files
    }

# Example usage:
# results = validate_pkl_columns_features('your_folder')

# Total CSV Record Counter
Calculate the total number of records across all CSV files in a folder.

In [ ]:
import pandas as pd
import os
from datetime import datetime

def calculate_total_records(folder_path):
    """
    Calculate the total number of records across all CSV files in the specified folder
    """
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"Folder {folder_path} does not exist!")
        return

    # Get all CSV files in the folder
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

    if not csv_files:
        print("No CSV files found in the folder!")
        return

    print(f"Found {len(csv_files)} CSV files to process...")

    total_records = 0

    for csv_file in csv_files:
        file_path = os.path.join(folder_path, csv_file)

        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            record_count = len(df)
            total_records += record_count

            print(f"Processed {csv_file}: {record_count} records")

        except Exception as e:
            print(f"Error processing {csv_file}: {str(e)}")

    print(f"\nTotal records across all CSV files: {total_records}")
    return total_records

# Example usage:
# Set the folder path - change this to your actual folder path
folder_path = r"1d-max"

# Calculate total records
total = calculate_total_records(folder_path)
print("Processing complete!")

# Environment Feature Inspection Script
Inspect features and columns in a gym trading environment.

In [3]:
import gym_trading_env
import gymnasium as gym
import pandas as pd
import numpy as np
from reward import reward_function_5 as custom_reward_function

def preprocess(df: pd.DataFrame):
    try:
        df["feature_macd"] = df["macd"]
        print(f"Successfully added feature_macd from macd column")
    except Exception as e:
        print(f"Error during preprocessing: {e}")
        print(f"Available columns: {df.columns.tolist()}")
    return df

def inspect_environment():
    print("=" * 60)
    print("ENVIRONMENT FEATURE INSPECTION")
    print("=" * 60)

    env = gym.make('MultiDatasetTradingEnv',
                   dataset_dir='dataset/1d-2005/train/*.pkl',
                   reward_function=custom_reward_function,
                   preprocess=preprocess,
                   verbose=2
                   )

    obs, info = env.reset(seed=42)

    print(f"\n1. DATASET INFO:")
    print(f"   Current dataset: {env.name}")
    print(f"   DataFrame shape: {env.df.shape}")

    print(f"\n2. COLUMN ANALYSIS:")
    print(f"   All columns: {env.df.columns.tolist()}")
    print(f"   Feature columns: {env._features_columns}")
    print(f"   Info columns: {env._info_columns}")

    print(f"\n3. MACD FEATURE CHECK:")
    if 'macd' in env.df.columns:
        print(f"   ✓ 'macd' column found in DataFrame")
        print(f"   MACD sample values: {env.df['macd'].head().tolist()}")
    else:
        print(f"   ✗ 'macd' column NOT found in DataFrame")

    if 'feature_macd' in env.df.columns:
        print(f"   ✓ 'feature_macd' column found in DataFrame")
        print(f"   Feature MACD sample values: {env.df['feature_macd'].head().tolist()}")
    else:
        print(f"   ✗ 'feature_macd' column NOT found in DataFrame")

    print(f"\n4. OBSERVATION SPACE:")
    print(f"   Observation space: {env.observation_space}")
    print(f"   Number of features: {env._nb_features}")
    print(f"   Static features: {env._nb_static_features}")

    print(f"\n5. ACTUAL OBSERVATION:")
    print(f"   Observation shape: {obs.shape}")
    print(f"   Observation sample: {obs}")

    print(f"\n6. FEATURE MAPPING:")
    for i, col in enumerate(env._features_columns):
        print(f"   Feature {i}: {col}")

    print(f"\n7. ENVIRONMENT STEP TEST:")
    for step in range(3):
        action = env.action_space.sample()
        obs, reward, done, truncated, info = env.step(action)
        print(f"   Step {step+1}: Action={action}, Reward={reward:.6f}, Done={done}, Truncated={truncated}")
        if done or truncated:
            break

    env.close()
    print("\n" + "=" * 60)
    print("INSPECTION COMPLETE")
    print("=" * 60)

inspect_environment()

ENVIRONMENT FEATURE INSPECTION
Successfully added feature_macd from macd column
Successfully added feature_macd from macd column
Selected dataset XLV_USD-1d-max.pkl ...

1. DATASET INFO:
   Current dataset: XLV_USD-1d-max.pkl
   DataFrame shape: (4027, 98)

2. COLUMN ANALYSIS:
   All columns: ['open', 'high', 'low', 'close', 'volume', 'symbol', 'rate', 'middle', 'tp', 'boll', 'boll_ub', 'boll_lb', 'macd', 'macds', 'macdh', 'pvo', 'pvos', 'pvoh', 'ppo', 'ppos', 'ppoh', 'qqe', 'qqel', 'qqes', 'cr', 'cr-ma1', 'cr-ma2', 'cr-ma3', 'tr', 'dx', 'adx', 'adxr', 'log-ret', 'wt1', 'wt2', 'supertrend_ub', 'supertrend_lb', 'supertrend', 'bop', 'cti', 'eribull', 'eribear', 'rvgi', 'rvgis', 'kst', 'num', 'ao', 'aroon', 'atr', 'cci', 'change', 'chop', 'cmo', 'coppock', 'dma', 'ichimoku', 'inertia', 'ftr', 'kama', 'kdjk', 'kdjd', 'kdjj', 'ker', 'mfi', 'ndi', 'pdi', 'pgo', 'psl', 'rsi', 'rsv', 'stochrsi', 'tema', 'trix', 'wr', 'vr', 'vwma', 'close_10_ema', 'close_10_sma', 'norm_close', 'norm_open', 'nor